In [ ]:
!pip install -q ultralytics==8.4.114 kagglehub pyyaml

import platform
import torch
import ultralytics

print("Python:", platform.python_version())
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到GPU。请到：运行时 → 更改运行时类型 → L4 GPU。"
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU显存:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ),
    "GB"
)

!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/fire_smoke_project")
RUNS_ROOT = DRIVE_ROOT / "runs"
DATA_ROOT = Path("/content/datasets/fire_smoke")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print("项目目录：", DRIVE_ROOT)
print("结果目录：", RUNS_ROOT)
print("临时数据目录：", DATA_ROOT)

In [ ]:
from pathlib import Path
import shutil
import kagglehub

DATASET_HANDLE = "sayedgamal99/smoke-fire-detection-yolo"
DATA_ROOT = Path("/content/datasets/fire_smoke")

downloaded_path = Path(
    kagglehub.dataset_download(DATASET_HANDLE)
)

print("Kaggle数据位置：", downloaded_path)
print("来源是否存在：", downloaded_path.exists())

if not downloaded_path.exists():
    raise FileNotFoundError("Kaggle数据集下载失败。")

# Remove an incomplete temporary copy from the current runtime.
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)

# Copy the dataset to local runtime storage for faster training I/O.
shutil.copytree(
    downloaded_path,
    DATA_ROOT,
    dirs_exist_ok=True
)

print("数据已复制到：", DATA_ROOT)
print("文件和文件夹数量：", len(list(DATA_ROOT.rglob("*"))))

In [ ]:
from pathlib import Path
import yaml

TRAIN_IMAGES = DATA_ROOT / "data/train/images"
VAL_IMAGES = DATA_ROOT / "data/val/images"
TEST_IMAGES = DATA_ROOT / "data/test/images"

TRAIN_LABELS = DATA_ROOT / "data/train/labels"
VAL_LABELS = DATA_ROOT / "data/val/labels"
TEST_LABELS = DATA_ROOT / "data/test/labels"

print("训练图片目录：", TRAIN_IMAGES.exists())
print("验证图片目录：", VAL_IMAGES.exists())
print("测试图片目录：", TEST_IMAGES.exists())

print("训练标签目录：", TRAIN_LABELS.exists())
print("验证标签目录：", VAL_LABELS.exists())
print("测试标签目录：", TEST_LABELS.exists())

required_paths = [
    TRAIN_IMAGES,
    VAL_IMAGES,
    TEST_IMAGES,
    TRAIN_LABELS,
    VAL_LABELS,
    TEST_LABELS,
]

missing_paths = [str(path) for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "以下数据路径不存在：\n" + "\n".join(missing_paths)
    )

ORIGINAL_YAML = DATA_ROOT / "data.yaml"

if not ORIGINAL_YAML.exists():
    yaml_candidates = sorted(DATA_ROOT.rglob("*.yaml"))

    if not yaml_candidates:
        raise FileNotFoundError("数据集中没有找到原始data.yaml。")

    ORIGINAL_YAML = yaml_candidates[0]

with open(ORIGINAL_YAML, "r", encoding="utf-8") as file:
    original_cfg = yaml.safe_load(file)

if "names" not in original_cfg:
    raise KeyError("原始data.yaml中没有names类别信息。")

fixed_cfg = {
    "train": str(TRAIN_IMAGES.resolve()),
    "val": str(VAL_IMAGES.resolve()),
    "test": str(TEST_IMAGES.resolve()),
    "names": original_cfg["names"],
}

DATA_YAML = DATA_ROOT / "data_colab.yaml"

with open(DATA_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        fixed_cfg,
        file,
        allow_unicode=True,
        sort_keys=False
    )

print("\n生成的YAML路径：", DATA_YAML)
print(DATA_YAML.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

def count_images(folder: Path) -> int:
    return sum(
        1
        for path in folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

def count_labels(folder: Path) -> int:
    return sum(
        1
        for path in folder.rglob("*.txt")
        if path.is_file()
    )

train_count = count_images(TRAIN_IMAGES)
val_count = count_images(VAL_IMAGES)
test_count = count_images(TEST_IMAGES)

print("训练图片数：", train_count)
print("验证图片数：", val_count)
print("测试图片数：", test_count)

print("训练标签文件数：", count_labels(TRAIN_LABELS))
print("验证标签文件数：", count_labels(VAL_LABELS))
print("测试标签文件数：", count_labels(TEST_LABELS))

print("类别顺序：", fixed_cfg["names"])
print("当前YAML：", DATA_YAML)

if min(train_count, val_count, test_count) == 0:
    raise RuntimeError("至少一个数据划分的图片数量为0，不能开始训练。")

In [ ]:
YOLO11_RUN_NAME = "yolo11s_640_100e_b16_s42"
YOLO11_RUN_DIR = RUNS_ROOT / YOLO11_RUN_NAME

print("YOLO11结果目录：", YOLO11_RUN_DIR)
print("该目录是否已存在：", YOLO11_RUN_DIR.exists())

if YOLO11_RUN_DIR.exists():
    raise FileExistsError(
        "这个实验目录已经存在。不要覆盖旧结果。"
        "请先检查其中是否已经有训练文件。"
    )

In [ ]:
from ultralytics import YOLO

MODEL_FILE = "yolo11s.pt"
RUN_NAME = "yolo11s_640_100e_b16_s42"

yolo11_model = YOLO(MODEL_FILE)

yolo11_results = yolo11_model.train(
    data=str(DATA_YAML),

    # Core parameters used for the four-model comparison.
    imgsz=640,
    epochs=100,
    batch=16,
    seed=42,

    # GPU and data-loading settings.
    device=0,
    workers=2,

    # Output directory.
    project=str(RUNS_ROOT),
    name=RUN_NAME,
    exist_ok=False,

    # Training configuration.
    pretrained=True,
    optimizer="auto",
    deterministic=True,
    amp=True,

    # Formal 100-epoch training run.
    patience=100,
    close_mosaic=10,

    # Save experiment outputs.
    save=True,
    save_period=5,
    plots=True,
    cache=False,
    verbose=True,
)

In [ ]:
!pip install -q ultralytics==8.4.114 kagglehub pyyaml

import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import shutil
import yaml
import kagglehub

DATASET_HANDLE = "sayedgamal99/smoke-fire-detection-yolo"

DATA_ROOT = Path("/content/datasets/fire_smoke")
DATA_YAML = DATA_ROOT / "data_colab.yaml"

# 1. Download the same dataset used for training.
downloaded_path = Path(
    kagglehub.dataset_download(DATASET_HANDLE)
)

print("数据集来源：", downloaded_path)
print("来源存在：", downloaded_path.exists())

if not downloaded_path.exists():
    raise FileNotFoundError("数据集下载失败。")

# 2. Restore the dataset to the training-time working path.
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)

shutil.copytree(
    downloaded_path,
    DATA_ROOT,
    dirs_exist_ok=True
)

# 3. Resolve train/validation/test paths.
TRAIN_IMAGES = DATA_ROOT / "data/train/images"
VAL_IMAGES = DATA_ROOT / "data/val/images"
TEST_IMAGES = DATA_ROOT / "data/test/images"

print("训练集：", TRAIN_IMAGES.exists())
print("验证集：", VAL_IMAGES.exists())
print("测试集：", TEST_IMAGES.exists())

if not TRAIN_IMAGES.exists():
    raise FileNotFoundError(f"找不到训练集：{TRAIN_IMAGES}")

if not VAL_IMAGES.exists():
    raise FileNotFoundError(f"找不到验证集：{VAL_IMAGES}")

if not TEST_IMAGES.exists():
    raise FileNotFoundError(f"找不到测试集：{TEST_IMAGES}")

# 4. Read class names from the dataset configuration.
ORIGINAL_YAML = DATA_ROOT / "data.yaml"

if not ORIGINAL_YAML.exists():
    yaml_candidates = [
        path
        for path in DATA_ROOT.rglob("*.yaml")
        if path.name != "data_colab.yaml"
    ]

    if not yaml_candidates:
        raise FileNotFoundError("没有找到原始data.yaml。")

    ORIGINAL_YAML = yaml_candidates[0]

with open(ORIGINAL_YAML, "r", encoding="utf-8") as file:
    original_cfg = yaml.safe_load(file)

# 5. Generate a Colab-specific dataset YAML.
fixed_cfg = {
    "train": str(TRAIN_IMAGES.resolve()),
    "val": str(VAL_IMAGES.resolve()),
    "test": str(TEST_IMAGES.resolve()),
    "names": original_cfg["names"],
}

with open(DATA_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        fixed_cfg,
        file,
        allow_unicode=True,
        sort_keys=False
    )

print("\n生成的YAML：")
print(DATA_YAML.read_text(encoding="utf-8"))

print("YAML存在：", DATA_YAML.exists())
print("测试集存在：", TEST_IMAGES.exists())

In [ ]:
from pathlib import Path
from ultralytics import YOLO

RUN_DIR = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolo11s_640_100e_b16_s42"
)

BEST_WEIGHT = RUN_DIR / "weights" / "best.pt"

print("best.pt存在：", BEST_WEIGHT.exists())
print("data.yaml存在：", DATA_YAML.exists())

if not BEST_WEIGHT.exists():
    raise FileNotFoundError("没有找到YOLO11的best.pt。")

model = YOLO(str(BEST_WEIGHT))

test_metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project=str(RUN_DIR.parent),
    name="yolo11s_final_test",
    exist_ok=True,
    plots=True,
)

In [ ]:
from pathlib import Path
import json
import platform
import subprocess
import sys
import shutil
import torch
import ultralytics

RUN_DIR = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolo11s_640_100e_b16_s42"
)

BEST_WEIGHT = RUN_DIR / "weights" / "best.pt"

summary = {
    "model": "YOLO11-S",
    "evaluation_split": "test",
    "training_epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "seed": 42,
    "precision": float(test_metrics.box.mp),
    "recall": float(test_metrics.box.mr),
    "mAP50": float(test_metrics.box.map50),
    "mAP50_95": float(test_metrics.box.map),
    "weight_size_mb": round(
        BEST_WEIGHT.stat().st_size / 1024**2, 3
    ),
    "parameters": int(
        sum(p.numel() for p in model.model.parameters())
    ),
    "speed_ms_per_image": test_metrics.speed,
    "per_class": {
        model.names[i]: {
            "precision": float(test_metrics.box.p[i]),
            "recall": float(test_metrics.box.r[i]),
            "mAP50": float(test_metrics.box.ap50[i]),
            "mAP50_95": float(test_metrics.box.maps[i]),
        }
        for i in range(len(test_metrics.box.maps))
    },
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda,
    "ultralytics": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0),
}

SUMMARY_PATH = RUN_DIR / "final_test_summary.json"
ENV_PATH = RUN_DIR / "environment.json"
FREEZE_PATH = RUN_DIR / "pip_freeze.txt"
YAML_COPY_PATH = RUN_DIR / "data_colab_used.yaml"

SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

environment = {
    "model": "YOLO11-S",
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "ultralytics": ultralytics.__version__,
}

ENV_PATH.write_text(
    json.dumps(environment, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

with open(FREEZE_PATH, "w", encoding="utf-8") as file:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=file,
        text=True,
        check=True,
    )

shutil.copy2(DATA_YAML, YAML_COPY_PATH)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\n测试摘要：", SUMMARY_PATH)
print("环境信息：", ENV_PATH)
print("依赖列表：", FREEZE_PATH)
print("数据配置：", YAML_COPY_PATH)